In [ ]:
import dlib
from scipy.spatial import distance
import cv2
import imutils
from imutils import face_utils
import numpy as np
import time
import pygame
import os
os.chdir(r"C:\Users\DT1\Desktop\proj-python\models")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
model="shape_predictor_68_face_landmarks.dat"
alarm=r"C:\Users\DT1\Desktop\proj-python\music\alarm_1.mp3"
def sound_alarm(path):
   pygame.mixer.init()
   pygame.mixer.music.load(path)
   pygame.mixer.music.play()
   time.sleep(1)
   pygame.mixer.music.stop()
def eye_aspect_ratio(eye):
    A=distance.euclidean(eye[1],eye[5])
    B=distance.euclidean(eye[2],eye[4])
    C=distance.euclidean(eye[0],eye[3])
    eq=(A+B)/(C*2)
    return float(eq)
def detect():
  thresh=0.25
  frames=20
  alarm_on=False
  
  detector=dlib.get_frontal_face_detector()
  predictor=dlib.shape_predictor(model)

  lstart,lend=face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]
  rstart,rend=face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]
  cap=cv2.VideoCapture(0)
  flag=0
  while True:
    ret,frame=cap.read()
    if not ret:
        break
    frame=imutils.resize(frame,width=640,height=480)
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    rects=detector(gray,0)
    for rect in rects:
        shape=predictor(gray,rect)
        shape=face_utils.shape_to_np(shape)
        leftEye=shape[lstart:lend]
        rightEye=shape[rstart:rend]
        leftEar=eye_aspect_ratio(leftEye)
        rightEar=eye_aspect_ratio(rightEye)
        ear=(leftEar+rightEar)/2.0
        leftEyeHull=cv2.convexHull(leftEye)
        rightEyeHull=cv2.convexHull(rightEye)
        cv2.drawContours(frame,[leftEyeHull],-1,(0,255,0),1)
        cv2.drawContours(frame,[rightEyeHull],-1,(0,255,0),1)
        if ear<thresh:
            flag+=1
            if flag>=frames:
                if not alarm_on:
                    alarm_on=True
                    
                cv2.putText(frame,"DROWSINESS ALERT",(10,30),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
                sound_alarm(alarm)
                cv2.putText(frame,str(flag),(10,60),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
        else:
            flag=0
            alarm_on=False
        cv2.putText(frame,"EAR: {:.2f}".format(ear),(300,30),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
    cv2.imshow("Frame",frame)
    key=cv2.waitKey(1)&0xFF
    if key==ord("q"):
        break
  cv2.destroyAllWindows()
  cap.release()
if __name__=="__main__":
    detect()